In [0]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
def ingest_to_bronze(
    spark: SparkSession, 
    schema: StructType, 
    source_path: str, 
    target_path: str
) -> DataFrame:
    """
    Reads a raw CSV file, adds ingestion metadata, and saves it as a Delta table.
    """
    # 1. Read the raw data
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .schema(schema)
        .csv(source_path)
    )
    
    # 2. Add ingestion timestamp
    df = df.withColumn("ingest_timestamp", current_timestamp())
    
    # 3. Write to Delta in overwrite mode
    df.write.mode("overwrite").format("delta").save(target_path)
    
    # Useful for debugging
    print(f"Successfully ingested data to {target_path}")

    return df

In [0]:
def flatten_nested_column_silver(df: DataFrame, new_column_name: str, old_column_name: str, schema: StructType) -> DataFrame:
    """
    Takes a DataFrame with a nested column and returns a new DataFrame with the nested column exploded.
    """
    # 1. Add a column with the parsed JSON
    df_parsed = df.withColumn(
    new_column_name, 
    from_json(col(old_column_name), 
              schema, {"allowSingleQuotes": "true"})
    ).drop(old_column_name) # 2. Drop the old column

    # 3. Loop through the nested items and add a column for each one
    for field in schema.fieldNames():
        df_parsed = df_parsed.withColumn(f"{new_column_name}_{field}", col(new_column_name).getItem(field))
        print(f"Sucessfully added column {new_column_name}_{field}")

    # 4. Drop the nested column
    df_parsed = df_parsed.drop(new_column_name)

    return df_parsed

In [0]:
def flatten_dataframe(df: DataFrame) -> DataFrame:
    """
    Takes a DataFrame with nested columns and returns a new DataFrame with the nested columns exploded.
    """
    